# Paper Tables

Generate LaTeX tables for the paper.

In [10]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

## Helper Functions

In [11]:
def load_run(fpath):
    """Load results from a single run."""
    fpath = Path(fpath)
    if not fpath.exists():
        return None
    with open(fpath) as f:
        return [json.loads(l) for l in f]

def compute_unified_metrics(entries):
    """Compute metrics for unified approach (baseline, directed, nudged)."""
    total = len(entries)
    compiled = sum(1 for e in entries if e.get("success") or e.get("lean_verification", {}).get("success", False))
    correct = sum(1 for e in entries if e.get("correct"))
    correct_compiled = sum(1 for e in entries 
                          if (e.get("success") or e.get("lean_verification", {}).get("success", False)) 
                          and e.get("correct"))
    return {
        "total": total,
        "compiled": compiled,
        "correct": correct,
        "correct_compiled": correct_compiled,
        "compilation": compiled / total * 100 if total > 0 else 0,
        "accuracy_overall": correct / total * 100 if total > 0 else 0,
        "accuracy_compiled": correct_compiled / compiled * 100 if compiled > 0 else 0,
    }

def compute_twostage_metrics(entries):
    """Compute metrics for two-stage approach."""
    total = len(entries)
    s1_pass = sum(1 for e in entries if e.get("stage1_success"))
    s2_pass = sum(1 for e in entries if e.get("stage2_success"))
    correct = sum(1 for e in entries if e.get("correct"))
    correct_compiled = sum(1 for e in entries if e.get("stage2_success") and e.get("correct"))
    return {
        "total": total,
        "compiled": s2_pass,
        "correct": correct,
        "correct_compiled": correct_compiled,
        "stage1": s1_pass / total * 100 if total > 0 else 0,
        "stage2": s2_pass / total * 100 if total > 0 else 0,
        "accuracy_overall": correct / total * 100 if total > 0 else 0,
        "accuracy_compiled": correct_compiled / s2_pass * 100 if s2_pass > 0 else 0,
    }

def fmt(values):
    """Format mean±std for LaTeX."""
    mean = np.mean(values)
    std = np.std(values)
    return f"{mean:.1f}\\tiny{{±{std:.1f}}}"

# Configuration
CONDITIONS = [
    ("Baseline", "baseline"),
    ("Directed$_\\text{T}$", "bidir_true"),
    ("Directed$_\\text{F}$", "bidir_false"),
    ("Nudged$_\\text{T}$", "spooky_true"),
    ("Nudged$_\\text{F}$", "spooky_false"),
]

MODELS = [("GPT-5", "gpt-5"), ("DeepSeek-R1", "deepseek")]
DATASETS = [("FOLIO", "folio"), ("M-LogiEval", "multilogieval")]

## Table 1: FOLIO Results (mean±std over 3 runs)

- **Acc (Overall)**: correct / total (all cases)
- **Acc (Compiled)**: correct_compiled / compiled (only verified cases)

In [12]:
# FOLIO Results with both accuracy metrics
print("FOLIO RESULTS (mean±std over 3 runs)")
print("=" * 95)
print(f"{'Model':<14} {'Condition':<20} {'Compilation %':>15} {'Acc (Overall)':>15} {'Acc (Compiled)':>16}")
print("-" * 95)

for model, model_dir in MODELS:
    # Unified approaches
    for cond_name, cond in CONDITIONS:
        folio_comp, folio_acc_overall, folio_acc_compiled = [], [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/folio/{model_dir}/{cond}_run{run}/results.jsonl")
            if entries:
                m = compute_unified_metrics(entries)
                folio_comp.append(m["compilation"])
                folio_acc_overall.append(m["accuracy_overall"])
                folio_acc_compiled.append(m["accuracy_compiled"])
        
        fc = f"{np.mean(folio_comp):.1f}±{np.std(folio_comp):.1f}"
        fa_o = f"{np.mean(folio_acc_overall):.1f}±{np.std(folio_acc_overall):.1f}"
        fa_c = f"{np.mean(folio_acc_compiled):.1f}±{np.std(folio_acc_compiled):.1f}"
        
        print(f"{model:<14} {cond_name:<20} {fc:>15} {fa_o:>15} {fa_c:>16}")
    
    # Two-Stage (show S1 and S2 compilation)
    model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
    folio_s1, folio_s2, folio_acc_overall, folio_acc_compiled = [], [], [], []
    
    for run in [1, 2, 3]:
        entries = load_run(f"../results/folio/twostage/{model_name}_run{run}/results.jsonl")
        if entries:
            m = compute_twostage_metrics(entries)
            folio_s1.append(m["stage1"])
            folio_s2.append(m["stage2"])
            folio_acc_overall.append(m["accuracy_overall"])
            folio_acc_compiled.append(m["accuracy_compiled"])
    
    fs1 = f"{np.mean(folio_s1):.1f}±{np.std(folio_s1):.1f}"
    fs2 = f"{np.mean(folio_s2):.1f}±{np.std(folio_s2):.1f}"
    fa_o = f"{np.mean(folio_acc_overall):.1f}±{np.std(folio_acc_overall):.1f}"
    fa_c = f"{np.mean(folio_acc_compiled):.1f}±{np.std(folio_acc_compiled):.1f}"
    
    print(f"{model:<14} {'Two-Stage':<20} S1:{fs1} S2:{fs2}  {fa_o:>10} {fa_c:>16}")
    print()

FOLIO RESULTS (mean±std over 3 runs)
Model          Condition              Compilation %   Acc (Overall)   Acc (Compiled)
-----------------------------------------------------------------------------------------------
GPT-5          Baseline                    98.2±0.8        85.2±0.4         85.3±0.9
GPT-5          Directed$_\text{T}$         99.7±0.5        91.3±0.2         91.3±0.2
GPT-5          Directed$_\text{F}$         99.2±0.5        92.0±0.5         91.9±0.4
GPT-5          Nudged$_\text{T}$           99.5±0.4        92.1±0.4         92.1±0.4
GPT-5          Nudged$_\text{F}$           98.9±0.6        92.3±0.2         92.5±0.4
GPT-5          Two-Stage            S1:100.0±0.0 S2:81.6±1.9    72.7±4.3         69.9±4.7

DeepSeek-R1    Baseline                    94.6±1.1        86.4±0.6         86.6±1.2
DeepSeek-R1    Directed$_\text{T}$         95.1±1.1        91.0±0.5         91.2±0.5
DeepSeek-R1    Directed$_\text{F}$         92.1±1.5        91.1±0.4         91.8±0.8
DeepSeek-R1

## Table 2: Multi-LogiEval Results (mean±std over 3 runs)

- **Acc (Overall)**: correct / total (all cases)
- **Acc (Compiled)**: correct_compiled / compiled (only verified cases)

In [13]:
# Multi-LogiEval Results with both accuracy metrics
print("MULTI-LOGIEVAL RESULTS (mean±std over 3 runs)")
print("=" * 95)
print(f"{'Model':<14} {'Condition':<20} {'Compilation %':>15} {'Acc (Overall)':>15} {'Acc (Compiled)':>16}")
print("-" * 95)

for model, model_dir in MODELS:
    # Unified approaches
    for cond_name, cond in CONDITIONS:
        multi_comp, multi_acc_overall, multi_acc_compiled = [], [], []
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/multilogieval/{model_dir}/{cond}_run{run}/results.jsonl")
            if entries:
                m = compute_unified_metrics(entries)
                multi_comp.append(m["compilation"])
                multi_acc_overall.append(m["accuracy_overall"])
                multi_acc_compiled.append(m["accuracy_compiled"])
        
        mc = f"{np.mean(multi_comp):.1f}±{np.std(multi_comp):.1f}"
        ma_o = f"{np.mean(multi_acc_overall):.1f}±{np.std(multi_acc_overall):.1f}"
        ma_c = f"{np.mean(multi_acc_compiled):.1f}±{np.std(multi_acc_compiled):.1f}"
        
        print(f"{model:<14} {cond_name:<20} {mc:>15} {ma_o:>15} {ma_c:>16}")
    
    # Two-Stage (show S1 and S2 compilation)
    model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
    multi_s1, multi_s2, multi_acc_overall, multi_acc_compiled = [], [], [], []
    
    for run in [1, 2, 3]:
        entries = load_run(f"../results/multilogieval/twostage/{model_name}_run{run}/results.jsonl")
        if entries:
            m = compute_twostage_metrics(entries)
            multi_s1.append(m["stage1"])
            multi_s2.append(m["stage2"])
            multi_acc_overall.append(m["accuracy_overall"])
            multi_acc_compiled.append(m["accuracy_compiled"])
    
    ms1 = f"{np.mean(multi_s1):.1f}±{np.std(multi_s1):.1f}"
    ms2 = f"{np.mean(multi_s2):.1f}±{np.std(multi_s2):.1f}"
    ma_o = f"{np.mean(multi_acc_overall):.1f}±{np.std(multi_acc_overall):.1f}"
    ma_c = f"{np.mean(multi_acc_compiled):.1f}±{np.std(multi_acc_compiled):.1f}"
    
    print(f"{model:<14} {'Two-Stage':<20} S1:{ms1} S2:{ms2}  {ma_o:>10} {ma_c:>16}")
    print()

MULTI-LOGIEVAL RESULTS (mean±std over 3 runs)
Model          Condition              Compilation %   Acc (Overall)   Acc (Compiled)
-----------------------------------------------------------------------------------------------
GPT-5          Baseline                    99.3±0.9        71.7±2.1         72.2±2.7
GPT-5          Directed$_\text{T}$         99.3±0.9        85.7±1.2         85.6±1.2
GPT-5          Directed$_\text{F}$         99.3±0.5        83.7±1.2         83.6±1.3
GPT-5          Nudged$_\text{T}$           99.0±0.8        92.0±0.8         92.3±0.9
GPT-5          Nudged$_\text{F}$           99.7±0.5        85.3±1.2         85.3±1.2
GPT-5          Two-Stage            S1:100.0±0.0 S2:89.7±4.5    56.7±2.9         59.1±3.1

DeepSeek-R1    Baseline                    97.3±0.9        69.7±1.7         70.5±2.1
DeepSeek-R1    Directed$_\text{T}$         96.3±2.5        82.3±2.5         82.6±3.1
DeepSeek-R1    Directed$_\text{F}$         94.0±2.2        82.0±1.6         82.6±1.4
De

## Directional Results by GT Match

Reorganize Directed/Nudged results by whether proof direction matches ground truth:
- **matches GT**: bidir_true on GT=True/yes, bidir_false on GT=False/no
- **wrong dir**: bidir_true on GT=False/no/Uncertain, bidir_false on GT=True/yes/Uncertain

In [ ]:
# Directional results reorganized by GT match
def split_by_gt_match(entries_true, entries_false):
    """Split entries by whether proof direction matches GT."""
    matches_gt = []
    wrong_dir = []
    
    for e in entries_true:
        gt = str(e.get("ground_truth", "")).lower()
        if gt in ["true", "yes"]:
            matches_gt.append(e)
        else:  # false, no, uncertain -> wrong direction
            wrong_dir.append(e)
    
    for e in entries_false:
        gt = str(e.get("ground_truth", "")).lower()
        if gt in ["false", "no"]:
            matches_gt.append(e)
        else:  # true, yes, uncertain -> wrong direction
            wrong_dir.append(e)
    
    return matches_gt, wrong_dir

print("DIRECTIONAL RESULTS BY GT MATCH (mean±std over 3 runs)")
print("=" * 100)
print(f"{'Dataset':<12} {'Model':<14} {'Approach':<10} {'GT Match':<12} {'Compilation %':>14} {'Acc (Overall)':>14} {'Acc (Compiled)':>16}")
print("-" * 100)

for dataset, dataset_dir in DATASETS:
    for model, model_dir in MODELS:
        for label, approach in [("Directed", "bidir"), ("Nudged", "spooky")]:
            matches_comp, matches_acc_o, matches_acc_c = [], [], []
            wrong_comp, wrong_acc_o, wrong_acc_c = [], [], []
            
            for run in [1, 2, 3]:
                entries_true = load_run(f"../results/{dataset_dir}/{model_dir}/{approach}_true_run{run}/results.jsonl") or []
                entries_false = load_run(f"../results/{dataset_dir}/{model_dir}/{approach}_false_run{run}/results.jsonl") or []
                
                matches, wrong = split_by_gt_match(entries_true, entries_false)
                
                if matches:
                    m = compute_unified_metrics(matches)
                    matches_comp.append(m["compilation"])
                    matches_acc_o.append(m["accuracy_overall"])
                    matches_acc_c.append(m["accuracy_compiled"])
                if wrong:
                    m = compute_unified_metrics(wrong)
                    wrong_comp.append(m["compilation"])
                    wrong_acc_o.append(m["accuracy_overall"])
                    wrong_acc_c.append(m["accuracy_compiled"])
            
            # Matches GT
            if matches_comp:
                mc = f"{np.mean(matches_comp):.1f}±{np.std(matches_comp):.1f}"
                ma_o = f"{np.mean(matches_acc_o):.1f}±{np.std(matches_acc_o):.1f}"
                ma_c = f"{np.mean(matches_acc_c):.1f}±{np.std(matches_acc_c):.1f}"
                print(f"{dataset:<12} {model:<14} {label:<10} {'matches':<12} {mc:>14} {ma_o:>14} {ma_c:>16}")
            
            # Wrong direction
            if wrong_comp:
                wc = f"{np.mean(wrong_comp):.1f}±{np.std(wrong_comp):.1f}"
                wa_o = f"{np.mean(wrong_acc_o):.1f}±{np.std(wrong_acc_o):.1f}"
                wa_c = f"{np.mean(wrong_acc_c):.1f}±{np.std(wrong_acc_c):.1f}"
                print(f"{'':<12} {'':<14} {'':<10} {'wrong_dir':<12} {wc:>14} {wa_o:>14} {wa_c:>16}")
        print()

In [ ]:
# Conservative predictions analysis (Uncertain/Unknown predictions)
print("CONSERVATIVE PREDICTIONS ANALYSIS (Uncertain/Unknown)")
print("=" * 100)
print()
print("Conservative = Model predicted Uncertain (couldn't prove True or False)")
print()

CONDITIONS_ANALYSIS = [
    ("Baseline", "baseline"),
    ("Directed_T", "bidir_true"),
    ("Directed_F", "bidir_false"),
    ("Nudged_T", "spooky_true"),
    ("Nudged_F", "spooky_false"),
]

for dataset, dataset_dir in DATASETS:
    print(f"\n{'='*100}")
    print(f"DATASET: {dataset}")
    print(f"{'='*100}")
    print(f"{'Model':<14} {'Condition':<15} {'Unc Pred':>10} {'Correct':>10} {'Precision':>12} {'GT=Unc':>10}")
    print("-" * 70)
    
    for model, model_dir in MODELS:
        for cond_name, cond in CONDITIONS_ANALYSIS:
            total_unc_pred = 0
            correct_unc = 0
            gt_unc_total = 0
            
            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries:
                    continue
                for e in entries:
                    pred = str(e.get("prediction", "")).lower()
                    gt = str(e.get("ground_truth", "")).lower()
                    
                    # Normalize
                    if pred in ["uncertain", "unknown", "failure"]:
                        pred = "uncertain"
                    if gt in ["uncertain", "unknown"]:
                        gt = "uncertain"
                        gt_unc_total += 1
                    
                    if pred == "uncertain":
                        total_unc_pred += 1
                        if gt == "uncertain":
                            correct_unc += 1
            
            if total_unc_pred > 0:
                precision = correct_unc / total_unc_pred * 100
                print(f"{model:<14} {cond_name:<15} {total_unc_pred:>10} {correct_unc:>10} {precision:>11.1f}% {gt_unc_total:>10}")
            else:
                print(f"{model:<14} {cond_name:<15} {total_unc_pred:>10} {correct_unc:>10} {'N/A':>12} {gt_unc_total:>10}")
        
        # Two-Stage
        model_name = "deepseek-r1" if model_dir == "deepseek" else "gpt-5"
        total_unc_pred = 0
        correct_unc = 0
        gt_unc_total = 0
        
        for run in [1, 2, 3]:
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            if not entries:
                continue
            for e in entries:
                pred = str(e.get("prediction", "")).lower()
                gt = str(e.get("ground_truth", "")).lower()
                
                if pred in ["uncertain", "unknown", "failure"]:
                    pred = "uncertain"
                if gt in ["uncertain", "unknown"]:
                    gt = "uncertain"
                    gt_unc_total += 1
                
                if pred == "uncertain":
                    total_unc_pred += 1
                    if gt == "uncertain":
                        correct_unc += 1
        
        if total_unc_pred > 0:
            precision = correct_unc / total_unc_pred * 100
            print(f"{model:<14} {'Two-Stage':<15} {total_unc_pred:>10} {correct_unc:>10} {precision:>11.1f}% {gt_unc_total:>10}")
        else:
            print(f"{model:<14} {'Two-Stage':<15} {total_unc_pred:>10} {correct_unc:>10} {'N/A':>12} {gt_unc_total:>10}")
        print()

print("\nLegend:")
print("  Unc Pred = Number of Uncertain predictions made")
print("  Correct = Among Unc predictions, how many had GT=Uncertain")
print("  Precision = Correct / Unc Pred (how reliable are Uncertain predictions)")
print("  GT=Unc = Total cases with GT=Uncertain in dataset")

## Structured Error Table (Compiled Cases Only)

Full breakdown of wrong predictions by Ground Truth → Prediction for each condition.

In [14]:
# Structured error table - all wrong predictions by GT -> Pred, BY DATASET and MODEL

# Simple conditions (for display purposes)
CONDITIONS_SIMPLE = [
    ("Baseline", "baseline"),
    ("Directed_T", "bidir_true"),
    ("Directed_F", "bidir_false"),
    ("Nudged_T", "spooky_true"),
    ("Nudged_F", "spooky_false"),
]

all_wrong = []
all_compiled = []
all_total = []

# Regular conditions
for dataset, dataset_dir in DATASETS:
    for model, model_dir in MODELS:
        for cond_name, cond in CONDITIONS_SIMPLE:
            for run in [1, 2, 3]:
                entries = load_run(f"../results/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl")
                if not entries:
                    continue
                for e in entries:
                    all_total.append({"dataset": dataset, "model": model, "condition": cond_name})
                    lean_success = e.get("success") or e.get("lean_verification", {}).get("success", False)
                    if lean_success:
                        all_compiled.append({"dataset": dataset, "model": model, "condition": cond_name})
                    if not lean_success or e.get("correct", False):
                        continue
                    pred = str(e.get("prediction", "")).lower()
                    gt = str(e.get("ground_truth", "")).lower()
                    if gt in ["yes"]: gt = "true"
                    elif gt in ["no"]: gt = "false"
                    if pred in ["yes"]: pred = "true"
                    elif pred in ["no"]: pred = "false"
                    all_wrong.append({"dataset": dataset, "model": model, "condition": cond_name, "gt": gt.capitalize(), "pred": pred.capitalize()})

# Two-stage
for dataset, dataset_dir in DATASETS:
    for model_name in ["gpt-5", "deepseek-r1"]:
        model_display = "GPT-5" if model_name == "gpt-5" else "DeepSeek-R1"
        for run in [1, 2, 3]:
            entries = load_run(f"../results/{dataset_dir}/twostage/{model_name}_run{run}/results.jsonl")
            if not entries:
                continue
            for e in entries:
                all_total.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage"})
                if e.get("stage2_success", False):
                    all_compiled.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage"})
                if not e.get("stage2_success", False) or e.get("correct", False):
                    continue
                pred = str(e.get("prediction", "")).lower()
                gt = str(e.get("ground_truth", "")).lower()
                if gt in ["yes"]: gt = "true"
                elif gt in ["no"]: gt = "false"
                if pred in ["yes"]: pred = "true"
                elif pred in ["no"]: pred = "false"
                all_wrong.append({"dataset": dataset, "model": model_display, "condition": "Two-Stage", "gt": gt.capitalize(), "pred": pred.capitalize()})

df_wrong_all = pd.DataFrame(all_wrong)
df_compiled_model = pd.DataFrame(all_compiled)
df_total_model = pd.DataFrame(all_total)

error_types = ["T→F", "T→Unc", "T→Fail", "F→T", "F→Unc", "F→Fail", "Unc→T", "Unc→F"]
error_map = {
    "T→F": ("True", "False"), "T→Unc": ("True", "Uncertain"), "T→Fail": ("True", "Failure"),
    "F→T": ("False", "True"), "F→Unc": ("False", "Uncertain"), "F→Fail": ("False", "Failure"),
    "Unc→T": ("Uncertain", "True"), "Unc→F": ("Uncertain", "False")
}

print("STRUCTURED ERROR TABLE BY DATASET AND MODEL (GT → Pred)")
print("="*100)
print(f"Total wrong (compiled): {len(df_wrong_all)}")
print()

for dataset in ["FOLIO", "M-LogiEval"]:
    print(f"\n{'='*100}")
    print(f"DATASET: {dataset}")
    print(f"{'='*100}")
    
    for model_name in ["GPT-5", "DeepSeek-R1"]:
        mask_wrong = (df_wrong_all["dataset"] == dataset) & (df_wrong_all["model"] == model_name)
        mask_comp = (df_compiled_model["dataset"] == dataset) & (df_compiled_model["model"] == model_name)
        mask_total = (df_total_model["dataset"] == dataset) & (df_total_model["model"] == model_name)
        
        model_wrong = df_wrong_all[mask_wrong]
        model_compiled = df_compiled_model[mask_comp]
        model_total = df_total_model[mask_total]
        
        print(f"\n{model_name}: {len(model_wrong)} wrong / {len(model_compiled)} compiled / {len(model_total)} total")
        print("-"*100)
        
        rows = []
        for cond in ["Baseline", "Directed_T", "Directed_F", "Nudged_T", "Nudged_F", "Two-Stage"]:
            cond_df = model_wrong[model_wrong["condition"] == cond]
            cond_compiled = len(model_compiled[model_compiled["condition"] == cond])
            cond_total = len(model_total[model_total["condition"] == cond])
            row = {"Condition": cond, "Wrong/Comp/Tot": f"{len(cond_df)}/{cond_compiled}/{cond_total}"}
            for etype in error_types:
                gt_val, pred_val = error_map[etype]
                count = len(cond_df[(cond_df["gt"] == gt_val) & (cond_df["pred"] == pred_val)])
                row[etype] = count if count > 0 else "-"
            rows.append(row)
        
        error_df = pd.DataFrame(rows)
        print(error_df.to_string(index=False))

print("\n" + "="*100)
print("Legend: T=True, F=False, Unc=Uncertain, Fail=Failure (couldn't prove)")
print("Wrong/Comp/Tot = Wrong among compiled / Compiled / Total cases")

STRUCTURED ERROR TABLE BY DATASET AND MODEL (GT → Pred)
Total wrong (compiled): 1548


DATASET: FOLIO

GPT-5: 433 wrong / 3514 compiled / 3654 total
----------------------------------------------------------------------------------------------------
 Condition Wrong/Comp/Tot T→F T→Unc T→Fail F→T F→Unc F→Fail Unc→T Unc→F
  Baseline     88/598/609   -    34      -   8    33      -    12     1
Directed_T     53/607/609   -     -     30   9     -      -    14     -
Directed_F     49/604/609   9     -      -   -     -     32     -     8
  Nudged_T     48/606/609   -     -     16  13     -      -    19     -
  Nudged_F     45/602/609  10     -      -   -     -     18     -    17
 Two-Stage    150/497/609  11    23      -  10    16      -    10    80

DeepSeek-R1: 383 wrong / 3305 compiled / 3654 total
----------------------------------------------------------------------------------------------------
 Condition Wrong/Comp/Tot T→F T→Unc T→Fail F→T F→Unc F→Fail Unc→T Unc→F
  Baseline     77/57